# Snowflake Postgres → Snowflake: Mirror Lab

This notebook is the **Snowflake-side companion** to the hands-on lab. Keep two windows open:

| Terminal (your laptop) | This notebook (Snowsight) |
|---|---|
| `task lab-N-...` changes data in **Postgres** and waits for Snowflake to catch up | Explore the same change from the **Snowflake** side |

Each section below is labeled with the lab step it follows. Nothing here writes to Postgres.

| Object | Name |
|---|---|
| Postgres instance | `{{ pg_instance_name }}` |
| Mirror | `{{ mirror_name }}` |
| Target database (created by the mirror) | `{{ target_database_name }}` |
| Lab database | `{{ database_name }}.{{ schema_name }}` |

> Run `task lab-1-create-mirror` before section 1: until then the target database does not exist.

In [ ]:
-- Set the context (Notebooks in Workspaces do not inherit a database or schema)
USE ROLE {{ admin_role_name }};
USE WAREHOUSE {{ warehouse_name }};
USE DATABASE {{ database_name }};
USE SCHEMA {{ schema_name }};

## 1. Mirror status  *(after `task lab-1-create-mirror`)*

`CREATE_MIRROR` created a publication and replication slot in Postgres, plus a target database, a serverless apply task, and one target table with `$changes` and `$live` companions per source table in Snowflake.

`LIST_MIRRORED_TABLES` is the per-table state: `SNAPSHOTTING` during the initial copy, `REPLICATING` once caught up.

In [ ]:
SELECT * FROM TABLE(SNOWFLAKE.POSTGRES.LIST_MIRRORS('{{ pg_instance_name }}'));

In [ ]:
-- apply_lag_seconds = how far the target tables trail what Postgres has flushed (0 = caught up)
SELECT status, refresh_interval, apply_lag_seconds, apply_lag_operations,
       last_applied_commit_time, last_operation_time, error_message
FROM TABLE(SNOWFLAKE.POSTGRES.DESCRIBE_MIRROR('{{ mirror_name }}'));

In [ ]:
CALL SNOWFLAKE.POSTGRES.LIST_MIRRORED_TABLES('{{ mirror_name }}');

In [ ]:
-- Each source table has a target table and a $changes table (an Iceberg table written by Postgres)...
SHOW TABLES IN SCHEMA {{ target_database_name }}.{{ pg_schema_name }};

In [ ]:
-- ...and a $live view: the target table plus changes not yet applied
SHOW VIEWS IN SCHEMA {{ target_database_name }}.{{ pg_schema_name }};

## 2. Base tables vs `$live`  *(after lab-1, and again after `task lab-2-add-data`)*

The **target table** is transactionally consistent and moves on every `refresh_interval`. The **`$live` view** shows new Postgres commits within about 30 seconds, whatever the interval. Run this cell right after lab-2 inserts 500 readings: `$live` reaches 985 before `READINGS` does.

In [ ]:
SELECT 'DEVICES' AS table_name,
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES) AS target_rows,
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES$live) AS live_rows
UNION ALL
SELECT 'SENSORS',
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.SENSORS),
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.SENSORS$live)
UNION ALL
SELECT 'READINGS',
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.READINGS),
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.READINGS$live);

## 3. Schema evolution  *(after `task lab-3-schema-changes`)*

Postgres added `STATUS` and renamed `LOCATION` to `SITE_LOCATION`. The mirror applied both without any reconfiguration. DDL is applied promptly, regardless of `refresh_interval`.

In [ ]:
DESCRIBE TABLE {{ target_database_name }}.{{ pg_schema_name }}.DEVICES;

In [ ]:
SELECT * FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES ORDER BY DEVICE_ID;

## 4. Deletes and the change feed  *(after `task lab-4-deletes`)*

The deleted readings are gone from `READINGS`, but `READINGS$changes` keeps a 7-day history:

| `_CHANGE_TYPE` | Meaning |
|---|---|
| `S` | Row from the initial snapshot |
| `I` | Insert (`_IS_UPDATE = TRUE` when it is the new row of an update) |
| `D` | Delete, or the old row of an update (same `_XID` as the update's `I` row) |


In [ ]:
SELECT _CHANGE_TYPE, _IS_UPDATE, COUNT(*) AS change_rows
FROM {{ target_database_name }}.{{ pg_schema_name }}.READINGS$changes
GROUP BY _CHANGE_TYPE, _IS_UPDATE
ORDER BY _CHANGE_TYPE, _IS_UPDATE;

## 5. `$live` and `$changes`  *(after `task lab-5-live-and-changes`)*

Lab-5 inserted `IoT-Gateway-Urgent` and set `IoT-Gateway-1` to `maintenance`. Query `$live` right away: both changes show up before the next apply run.

> `$live` is **not transactional**: it can expose part of a source transaction. Query the target tables when you need consistency.

In [ ]:
SELECT * FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES$live ORDER BY CREATED_AT DESC LIMIT 5;

In [ ]:
-- A gap means changes are waiting for the next apply run
SELECT (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES) AS devices,
       (SELECT COUNT(*) FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES$live) AS devices_live;

In [ ]:
-- An UPDATE arrives as a D/I pair sharing one _XID. The lab-3 column DDL started a
-- fresh DEVICES$changes feed, so only changes made after the rename appear here.
SELECT _COMMIT_TIME, _XID, _CHANGE_TYPE, _IS_UPDATE, DEVICE_ID, DEVICE_NAME, STATUS
FROM {{ target_database_name }}.{{ pg_schema_name }}.DEVICES$changes
ORDER BY _COMMIT_LSN, _LSN;

## 6. Operate and monitor  *(during and after `task lab-6-operate-mirror`)*

Lab-6 shortens the refresh interval with `ALTER_MIRROR`, adds the new `maintenance_events` table, then suspends and resumes the mirror. Every apply run is a serverless task named `APPLY_MIRROR_{{ mirror_name }}`; its compute is the main mirroring cost.

In [ ]:
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE name ILIKE 'APPLY_MIRROR_{{ mirror_name }}'
ORDER BY scheduled_time DESC
LIMIT 20;

In [ ]:
CALL SNOWFLAKE.POSTGRES.QUERY_ADMIN_LOG(
    mirror_name       => '{{ mirror_name }}',
    postgres_instance => '{{ pg_instance_name }}',
    level             => 'INFO',
    since_ts          => NULL,
    max_rows          => 20
);

In [ ]:
-- The table added with ALTER_MIRROR(add_tables => ...)
SELECT * FROM {{ target_database_name }}.{{ pg_schema_name }}.MAINTENANCE_EVENTS ORDER BY EVENT_ID;

## 7. Analytics on mirrored data  *(after `task lab-7-analytics`)*

Mirrored tables are ordinary Snowflake tables with change tracking enabled, so you can build on them directly:

- **Dynamic table** `{{ dynamic_table_name }}`: hourly metrics per site, refreshed automatically
- **Semantic view** `{{ semantic_view_name }}`: business-friendly dimensions and metrics
- **Cortex Agent** `{{ agent_name }}`: natural-language questions over the semantic view (try it in **AI & ML > Agents**)

In [ ]:
SELECT SITE_LOCATION, SENSOR_TYPE, READING_HOUR, READING_COUNT, ROUND(AVG_VALUE, 2) AS AVG_VALUE
FROM {{ database_name }}.{{ schema_name }}.{{ dynamic_table_name }}
ORDER BY READING_HOUR DESC, SITE_LOCATION, SENSOR_TYPE
LIMIT 20;

In [ ]:
# Hourly average temperature per site, from the dynamic table
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
df = session.sql("""
    SELECT READING_HOUR, SITE_LOCATION, AVG_VALUE::FLOAT AS AVG_TEMPERATURE_C
    FROM {{ database_name }}.{{ schema_name }}.{{ dynamic_table_name }}
    WHERE SENSOR_TYPE = 'Temperature'
    ORDER BY READING_HOUR
""").to_pandas()

# One line per site
ax = df.pivot(index="READING_HOUR", columns="SITE_LOCATION", values="AVG_TEMPERATURE_C").plot(figsize=(10, 4))
ax.set_xlabel("Reading hour")
ax.set_ylabel("Avg temperature (C)")
plt.show()


In [ ]:
SELECT *
FROM SEMANTIC_VIEW(
    {{ database_name }}.{{ schema_name }}.{{ semantic_view_name }}
    DIMENSIONS DEVICES.SITE_LOCATION, SENSORS.SENSOR_TYPE
    METRICS READINGS.AVG_VALUE, READINGS.READING_COUNT
)
ORDER BY SITE_LOCATION, SENSOR_TYPE;

## 8. Least-privilege reads

Reading mirrored data is governed by the `{{ target_database_name }}_ADMIN` database role. Lab-1 granted it to `{{ analyst_role_name }}`, a role with no other privileges on the mirror.

In [ ]:
USE ROLE {{ analyst_role_name }};
SELECT COUNT(*) AS readings_visible_to_analyst FROM {{ target_database_name }}.{{ pg_schema_name }}.READINGS;

In [ ]:
USE ROLE {{ admin_role_name }};

## 9. Iceberg under the hood

Every `$changes` table is an **Apache Iceberg** table that Postgres writes through `pg_lake`, and Snowflake reads it with no copy step. With `MIRROR_TARGET_TABLE_TYPE=ICEBERG`, the target tables are Snowflake-managed Iceberg tables too.

In [ ]:
SHOW ICEBERG TABLES IN DATABASE {{ target_database_name }};

## Wrap-up

You replicated an operational Postgres database into Snowflake with **one procedure call**: no ETL pipeline, no connector, no external CDC service. Inserts, updates, deletes, and schema changes all flowed through.

When you're done, run `task demo-teardown` in the terminal. It drops the mirror (and its target database), the lab objects, and in create mode the Postgres instance. Target database names can never be reused, so bump `MIRROR_TARGET_DATABASE_NAME` before your next run.